# DriveIQ – Next Steps: LLM #1 Full Run + LLM #2

**Prerequisites (already done):**
- `dspydantic` installed, OpenAI API key set
- Pydantic validation confirmed for both road files
- LLM #1 tested on 10-window sample

**This notebook produces exactly 4 output files:**
- `processed_outputs/secondary/llm1_results_secondary.json`
- `processed_outputs/motor/llm1_results_motor.json`
- `processed_outputs/secondary/llm2_results_secondary.json`
- `processed_outputs/motor/llm2_results_motor.json`

## 0 — Imports & Environment

In [5]:
import os, json, time
from pathlib import Path
from typing import List, Dict, Any
from collections import defaultdict, Counter

from pydantic import BaseModel, ConfigDict, Field, field_validator
from dspydantic import Prompter

# ── API key ───────────────────────────────────────────────────────────────────
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY_HERE"
print("OPENAI_API_KEY set?", bool(os.getenv("OPENAI_API_KEY")))

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR       = Path("JSON files")
SECONDARY_PATH = Path("secondary_output.json")
MOTOR_PATH     = Path("motor_output.json")



OUT_SEC = Path("processed_outputs/secondary")
OUT_MOT = Path("processed_outputs/motor")
OUT_SEC.mkdir(parents=True, exist_ok=True)
OUT_MOT.mkdir(parents=True, exist_ok=True)

print("Output dirs ready.")

OPENAI_API_KEY set? True


Output dirs ready.


## 1 — Schemas, Loaders & In-Memory Preparation

In [6]:
# ── Pydantic models ───────────────────────────────────────────────────────────
class TriggerFeature(BaseModel):
    model_config = ConfigDict(extra="forbid")
    feature: str = Field(..., min_length=1)
    value: float
    unit: str | None = None

    @field_validator("feature")
    @classmethod
    def _strip(cls, v: str) -> str:
        return v.strip()

class WindowOutput(BaseModel):
    model_config = ConfigDict(extra="forbid")
    window_id: int = Field(..., ge=0)
    predicted_label: str
    alert_cause: str
    severity: float
    knn_distance: float
    trigger_features: List[TriggerFeature] = Field(default_factory=list)

class BaseSession(BaseModel):
    model_config = ConfigDict(extra="forbid")
    session_id: int
    road_type: str
    performance_score: float
    windows: List[WindowOutput]

# ── Loader ────────────────────────────────────────────────────────────────────
def load_sessions(path: Path) -> List[BaseSession]:
    raw = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(raw, list):
        raise TypeError("Expected a list of sessions")
    return [BaseSession.model_validate(item) for item in raw]

SECONDARY_PATH = Path("JSON files/secondary_output.json")
MOTOR_PATH = Path("JSON files/motor_output.json")

print(f"Secondary: {len(secondary_sessions)} sessions")
print(f"Motor:     {len(motor_sessions)} sessions")

Secondary: 11 sessions
Motor:     18 sessions


In [7]:
# ── Flatten windows in memory (nothing written to disk) ───────────────────────
def flatten_sessions(sessions: List[BaseSession], road_tag: str) -> List[Dict[str, Any]]:
    flat = []
    for s in sessions:
        for w in s.windows:
            flat.append({
                "road_type":         road_tag,
                "session_id":        s.session_id,
                "window_id":         w.window_id,
                "window_key":        f"{road_tag}_s{s.session_id}_w{w.window_id}",
                "predicted_label":   w.predicted_label,
                "alert_cause":       w.alert_cause,
                "severity":          float(w.severity),
                "knn_distance":      float(w.knn_distance),
                "trigger_features":  [tf.model_dump() for tf in w.trigger_features],
                "performance_score": s.performance_score,
            })
    return flat

sec_flat = flatten_sessions(secondary_sessions, "Secondary")
mot_flat = flatten_sessions(motor_sessions,     "Motorway")

# ── Filter to abnormal only (in memory) ──────────────────────────────────────
sec_abnormal = [w for w in sec_flat if w["predicted_label"] in ("Aggressive", "Drowsy")]
mot_abnormal = [w for w in mot_flat if w["predicted_label"] in ("Aggressive", "Drowsy")]

print(f"Secondary abnormal windows: {len(sec_abnormal)}")
print(f"Motor     abnormal windows: {len(mot_abnormal)}")

Secondary abnormal windows: 215
Motor     abnormal windows: 369

## 2 — LLM #1: Window-Level Feedback

Writes only:
- `processed_outputs/secondary/llm1_results_secondary.json`
- `processed_outputs/motor/llm1_results_motor.json`

**Resumable** — already-processed `window_key`s are skipped on re-run.

In [8]:
class WindowFeedback(BaseModel):
    model_config = ConfigDict(extra="forbid")
    window_key: str
    feedback: str = Field(
        ...,
        description="2-4 sentences, single paragraph, evidence-based, no headings."
    )

window_prompter = Prompter(model=WindowFeedback, model_id="openai/gpt-4o-mini")

def build_window_prompt(item: Dict[str, Any]) -> str:
    feats = json.dumps(item["trigger_features"], ensure_ascii=False)
    return (
        "You are a professional driving mentor.\n"
        "Write concise, evidence-based feedback for the driver about ONE 4-minute window.\n"
        "Rules:\n"
        "  - Use ONLY the alert_cause and trigger_features listed below as evidence.\n"
        "  - Do NOT invent numbers, speeds, or behaviours not present in the features.\n"
        "  - 2-4 sentences, single paragraph, no headings, no lists.\n\n"
        f"window_key:       {item['window_key']}\n"
        f"predicted_label:  {item['predicted_label']}\n"
        f"alert_cause:      {item['alert_cause']}\n"
        f"severity:         {item['severity']:.3f}\n"
        f"trigger_features: {feats}\n"
    )

def run_llm1(
    abnormal_windows: List[Dict[str, Any]],
    out_path: Path,
    batch_size: int = 20,
    sleep_s: float = 0.5,
) -> List[Dict[str, Any]]:
    # Resume support
    if out_path.exists():
        results   = json.loads(out_path.read_text(encoding="utf-8"))
        done_keys = {r["window_key"] for r in results}
        print(f"  Resuming - {len(done_keys)} already done.")
    else:
        results, done_keys = [], set()

    remaining = [w for w in abnormal_windows if w["window_key"] not in done_keys]
    total = len(remaining)
    print(f"  Windows to process: {total}")

    for batch_start in range(0, total, batch_size):
        batch = remaining[batch_start : batch_start + batch_size]
        for item in batch:
            try:
                out = window_prompter.run(build_window_prompt(item))
                results.append(out.model_dump())
            except Exception as e:
                print(f"  Warning: Failed on {item['window_key']}: {e}")
                results.append({"window_key": item["window_key"], "feedback": f"ERROR: {e}"})

        # Save after each batch (crash-safe)
        out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
        processed = min(batch_start + batch_size, total)
        print(f"  [{processed}/{total}] saved")
        time.sleep(sleep_s)

    return results

print("LLM #1 ready.")

LLM #1 ready.


In [ ]:
print("=== LLM #1 - Secondary Road ===")
sec_llm1_results = run_llm1(
    abnormal_windows = sec_abnormal,
    out_path         = OUT_SEC / "llm1_results_secondary.json",
)
print(f"Done - {len(sec_llm1_results)} results")

=== LLM #1 - Secondary Road ===
  Windows to process: 215
  [20/215] saved
  [40/215] saved
  [60/215] saved


In [ ]:
print("=== LLM #1 - Motor Road ===")
mot_llm1_results = run_llm1(
    abnormal_windows = mot_abnormal,
    out_path         = OUT_MOT / "llm1_results_motor.json",
)
print(f"Done - {len(mot_llm1_results)} results")

=== LLM #1 - Motor Road ===
  Windows to process: 369
  [20/369] saved
  [40/369] saved
  [60/369] saved
  [80/369] saved
  [100/369] saved
  [120/369] saved
  [140/369] saved
  [160/369] saved
  [180/369] saved
  [200/369] saved
  [220/369] saved
  [240/369] saved
  [260/369] saved
  [280/369] saved
  [300/369] saved
  [320/369] saved
  [340/369] saved
  [360/369] saved


## 3 — LLM #2: Session-Level Overall Feedback

Writes only:
- `processed_outputs/secondary/llm2_results_secondary.json`
- `processed_outputs/motor/llm2_results_motor.json`

Each entry contains `session_key`, `student_feedback`, and `instructor_feedback`.

In [ ]:
class SessionFeedback(BaseModel):
    model_config = ConfigDict(extra="forbid")
    session_key:         str
    student_feedback:    str = Field(..., description="2-4 sentences addressed to the driver.")
    instructor_feedback: str = Field(..., description="2-4 sentences addressed to the instructor.")

session_prompter = Prompter(model=SessionFeedback, model_id="openai/gpt-4o-mini")

def tone_for_score(score: float) -> str:
    if score >= 80:   return "positive and encouraging - highlight strengths, minor notes only"
    elif score >= 60: return "balanced - acknowledge positives, clearly state areas for improvement"
    elif score >= 40: return "constructive but urgent - emphasise specific safety issues"
    else:             return "serious and direct - significant safety concerns, clear corrective actions required"

def build_session_summaries(
    sessions: List[BaseSession],
    flat: List[Dict[str, Any]],
    road_tag: str,
) -> List[Dict[str, Any]]:
    """Aggregate per-session stats in memory — nothing written to disk."""
    score_map  = {s.session_id: s.performance_score for s in sessions}
    by_session = defaultdict(list)
    for w in flat:
        by_session[w["session_id"]].append(w)

    summaries = []
    for sid, wins in sorted(by_session.items()):
        total    = len(wins)
        abnormal = sum(1 for w in wins if w["predicted_label"] in ("Aggressive", "Drowsy"))
        causes   = Counter(w["alert_cause"] for w in wins if w["alert_cause"])
        summaries.append({
            "session_key":       f"{road_tag}_s{sid}",
            "road_type":         road_tag,
            "session_id":        sid,
            "performance_score": score_map.get(sid, 50.0),
            "total_windows":     total,
            "abnormal_count":    abnormal,
            "top_causes":        [c for c, _ in causes.most_common(3)],
        })
    return summaries

def build_session_prompt(summary: Dict[str, Any]) -> str:
    score        = summary["performance_score"]
    top_causes   = ", ".join(summary["top_causes"]) or "None recorded"
    pct_abnormal = (
        round(100 * summary["abnormal_count"] / summary["total_windows"], 1)
        if summary["total_windows"] > 0 else 0
    )
    return (
        "You are DriveIQ, an AI driving mentor.\n"
        "Generate TWO short feedback paragraphs for a completed driving session.\n"
        f"Tone: {tone_for_score(score)}.\n\n"
        f"session_key:       {summary['session_key']}\n"
        f"road_type:         {summary['road_type']}\n"
        f"performance_score: {score:.1f}/100\n"
        f"total_windows:     {summary['total_windows']}\n"
        f"abnormal_windows:  {summary['abnormal_count']} ({pct_abnormal}%)\n"
        f"top_alert_causes:  {top_causes}\n\n"
        "Rules:\n"
        "  - student_feedback: 2-4 sentences, speak directly to the driver ('you').\n"
        "  - instructor_feedback: 2-4 sentences, speak to the instructor ('the student').\n"
        "  - No invented data beyond what is listed above.\n"
        "  - No bullet points or headings inside the paragraphs.\n"
    )

def run_llm2(
    summaries: List[Dict[str, Any]],
    out_path: Path,
    sleep_s: float = 0.5,
) -> List[Dict[str, Any]]:
    # Resume support
    if out_path.exists():
        results   = json.loads(out_path.read_text(encoding="utf-8"))
        done_keys = {r["session_key"] for r in results}
        print(f"  Resuming - {len(done_keys)} already done.")
    else:
        results, done_keys = [], set()

    remaining = [s for s in summaries if s["session_key"] not in done_keys]
    total = len(remaining)
    print(f"  Sessions to process: {total}")

    for i, summary in enumerate(remaining, 1):
        try:
            out = session_prompter.run(build_session_prompt(summary))
            results.append(out.model_dump())
        except Exception as e:
            print(f"  Warning: Failed on {summary['session_key']}: {e}")
            results.append({
                "session_key":         summary["session_key"],
                "student_feedback":    f"ERROR: {e}",
                "instructor_feedback": f"ERROR: {e}",
            })

        out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"  [{i}/{total}] {summary['session_key']} saved")
        time.sleep(sleep_s)

    return results

print("LLM #2 ready.")

LLM #2 ready.


In [ ]:
print("=== LLM #2 - Secondary Road ===")
sec_summaries    = build_session_summaries(secondary_sessions, sec_flat, "Secondary")
sec_llm2_results = run_llm2(
    summaries = sec_summaries,
    out_path  = OUT_SEC / "llm2_results_secondary.json",
)
print(f"Done - {len(sec_llm2_results)} sessions")

=== LLM #2 - Secondary Road ===
  Sessions to process: 11
  [1/11] Secondary_s0 saved
  [2/11] Secondary_s1 saved
  [3/11] Secondary_s2 saved
  [4/11] Secondary_s3 saved
  [5/11] Secondary_s4 saved
  [6/11] Secondary_s5 saved
  [7/11] Secondary_s6 saved
  [8/11] Secondary_s7 saved
  [9/11] Secondary_s8 saved
  [10/11] Secondary_s9 saved
  [11/11] Secondary_s10 saved
Done - 11 sessions


In [ ]:
print("=== LLM #2 - Motor Road ===")
mot_summaries    = build_session_summaries(motor_sessions, mot_flat, "Motorway")
mot_llm2_results = run_llm2(
    summaries = mot_summaries,
    out_path  = OUT_MOT / "llm2_results_motor.json",
)
print(f"Done - {len(mot_llm2_results)} sessions")

=== LLM #2 - Motor Road ===
  Sessions to process: 18
  [1/18] Motorway_s0 saved
  [2/18] Motorway_s1 saved
  [3/18] Motorway_s2 saved
  [4/18] Motorway_s3 saved
  [5/18] Motorway_s4 saved
  [6/18] Motorway_s5 saved
  [7/18] Motorway_s6 saved
  [8/18] Motorway_s7 saved
  [9/18] Motorway_s8 saved
  [10/18] Motorway_s9 saved
  [11/18] Motorway_s10 saved
  [12/18] Motorway_s11 saved
  [13/18] Motorway_s12 saved
  [14/18] Motorway_s13 saved
  [15/18] Motorway_s14 saved
  [16/18] Motorway_s15 saved
  [17/18] Motorway_s16 saved
  [18/18] Motorway_s17 saved
Done - 18 sessions


## 4 — Verify Outputs

In [ ]:
import pprint, json
from pathlib import Path

OUT_SEC = Path("processed_outputs/secondary")
OUT_MOT = Path("processed_outputs/motor")

files = [
    OUT_SEC / "llm1_results_secondary.json",
    OUT_MOT / "llm1_results_motor.json",
    OUT_SEC / "llm2_results_secondary.json",
    OUT_MOT / "llm2_results_motor.json",
]

for f in files:
    data = json.loads(f.read_text(encoding="utf-8"))
    print(f"{f.name}: {len(data)} entries")

# Preview samples directly from disk
sec_llm1 = json.loads((OUT_SEC / "llm1_results_secondary.json").read_text())
sec_llm2 = json.loads((OUT_SEC / "llm2_results_secondary.json").read_text())

print("\n--- Sample LLM #1 entry ---")
pprint.pprint(sec_llm1[0])

print("\n--- Sample LLM #2 entry ---")
pprint.pprint(sec_llm2[0])

llm1_results_secondary.json: 215 entries
llm1_results_motor.json: 360 entries
llm2_results_secondary.json: 11 entries
llm2_results_motor.json: 18 entries

--- Sample LLM #1 entry ---
{'feedback': 'During the observed window, the driver exhibited aggressive '
             'driving behavior characterized by harsh acceleration and '
             'braking, as indicated by a vertical acceleration of 5.55 m/sÂ² '
             'and a horizontal acceleration of 5.25 m/sÂ². This level of '
             'acceleration suggests a concerning lack of control and '
             'consideration for safe driving practices, which could increase '
             'the risk of accidents.',
 'window_key': 'Secondary_s0_w0'}

--- Sample LLM #2 entry ---
{'instructor_feedback': 'The student demonstrated a concerning level of '
                        'performance this session with a score of 24.0 out of '
                        '100, displaying 100% abnormal windows. It is critical '
                        'to